In [0]:
from datetime import datetime, timezone
from delta.tables import DeltaTable
from pyspark.sql import functions as F

bronze_layer = "s3a://aws-scraping-test-oks/raw/products/"

today = datetime.now(timezone.utc).strftime("%Y-%m-%d")

In [0]:

# 1. Read raw JSON from S3 (Bronze) only for today's date

df_bronze = spark.read \
    .option("multiline", "true") \
    .option("basePath", bronze_layer) \
    .json(f"{bronze_layer}date={today}/")

# 2. Clean and cast data types
df_silver = df_bronze \
    .withColumn("price", df_bronze["price"].cast("double")) \
    .withColumn("scraped_at", df_bronze["scraped_at"].cast("timestamp")) \
    .withColumn("date", F.col("date").cast("date")) \
    .dropDuplicates(["sku", "scraped_at"])

In [0]:
# 3. Data quality checks --> Circuit Breaker
assert df_silver.filter("price IS NULL").count() == 0, "Знайдено рядки без ціни"
assert df_silver.filter("sku IS NULL").count()==0, "Знайдено рядки без коду СКЮ"
assert df_silver.count() == df_silver.select("sku", "scraped_at").distinct().count(), \
    "Знайдено дублікати (sku, scraped_at)"

In [0]:
# 4. Write to existing Delta table gold_db.silver_products using liquid clustering

from delta.tables import DeltaTable

target = DeltaTable.forName(spark, "gold_db.silver_products")

result = target.alias("t").merge(
    df_silver.alias("s"),
    f"t.sku = s.sku AND t.scraped_at = s.scraped_at AND t.date = DATE'{today}'"
).whenNotMatchedInsertAll().execute()

inserted = result.collect()[0]["num_inserted_rows"]
print(f"MERGE inserted {inserted} new rows into silver_products")

In [0]:
# 5. Data quality checks --> inserted rows>0 or there already rows with current date

rows_today = spark.sql(
    f"SELECT COUNT(*) FROM gold_db.silver_products WHERE date = DATE'{today}'"
).collect()[0][0]

assert inserted > 0 or rows_today > 0, \
    "MERGE inserted 0 rows and no data exists for today — possible scraping failure"

In [0]:
# %sql
# select *
# from gold_db.silver_products
# where date = date(current_date)

In [0]:
# df_bronze.printSchema()